# 🌲 YOLOv8n-seg 樹幹偵測模型訓練

**目標**：訓練一個只辨識 `tree_trunk`（樹幹）的實例分割模型

**匯出**：TFLite INT8 (~6.3MB) → Flutter 手機端 30+ FPS

---

## 使用前設定
1. 上方選單 → **執行階段** → **變更執行階段類型** → 選 **T4 GPU**（最划算，夠用）
   - T4 (16GB) → 訓練約 40-60 分鐘 ✅ 推薦
   - L4 (24GB) → 訓練約 20-30 分鐘（速度較快但較貴）
   - A100/H100 → 殺雞用牛刀，更貴但對 YOLOv8n 沒明顯加速
2. 按照 Cell 順序從上到下執行
3. 最後下載匯出的模型檔

## Step 0：確認 GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    total = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', None)
    if total:
        print(f'VRAM: {total / 1024**3:.1f} GB')
    else:
        print(f'VRAM: (see nvidia-smi output above)')

## Step 1：安裝依賴

In [ ]:
!pip install ultralytics roboflow supervision -q
print('✓ 安裝完成')

## Step 2：從 Roboflow 下載資料集

### 如何取得 API Key：
1. 前往 https://app.roboflow.com/settings/api
2. 複製 **Private API Key**
3. 貼到下方 `ROBOFLOW_API_KEY` 欄位

### 如何找樹幹資料集：
1. 前往 https://universe.roboflow.com
2. 搜尋 `tree trunk segmentation`
3. 選擇一個有 **Instance Segmentation** 標註的資料集
4. 點進去 → 記下 **workspace / project / version**

下方已預填一個公開資料集，你也可以換成自己找到的：

In [ ]:
#@title 設定 Roboflow 資料集 { display-mode: "form" }

#@markdown ### Roboflow 設定
ROBOFLOW_API_KEY = 'uwNljxzf8xgKGZ9Is3py' #@param {type:"string"}
#@markdown ---
#@markdown ### 資料集設定（預設用最佳公開樹幹資料集 1.3k 張 Instance Segmentation）
#@markdown 資料集來源: https://universe.roboflow.com/tree-trunks/tree-trunk-detection-bi-axe
#@markdown 類別: trunk, post, sprinkler → 訓練時會自動只保留 trunk
WORKSPACE = 'tree-trunks' #@param {type:"string"}
PROJECT = 'tree-trunk-detection-bi-axe' #@param {type:"string"}
VERSION = 1 #@param {type:"integer"}
#@markdown ---
#@markdown ### 備選資料集（如果上面的下不了，換這些試試）:
#@markdown - tree-trunks / tree-trunk-detection-369pz / v1 (1.1k images)
#@markdown - fxq / tree-ja0cc / v1 (1.1k images)
#@markdown - wurdataset / tree-trunk-segmentation-ixblx / v1 (172 images, 小但精確)

if not ROBOFLOW_API_KEY:
    print('❌ 請填入 Roboflow API Key！')
    print('   前往 https://app.roboflow.com/settings/api 取得')
else:
    print(f'✓ API Key 已設定')
    print(f'✓ 資料集: {WORKSPACE}/{PROJECT} v{VERSION}')

In [ ]:
# 下載資料集
from roboflow import Roboflow
import os

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download(
    model_format='yolov8',
    location='/content/dataset',
    overwrite=True
)

# 確認下載結果
print(f'\n✓ 資料集下載完成：{dataset.location}')

# 檢查結構
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))])
        print(f'  {split}: {count} images')

## Step 3：修正 data.yaml（只保留 trunk 類別，移除 post/sprinkler 等）

In [ ]:
import yaml
import os

data_yaml_path = os.path.join(dataset.location, 'data.yaml')

# 讀取原始 data.yaml
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print('原始設定:')
original_names = data_config.get("names", [])
original_nc = data_config.get('nc', 1)
print(f'  Classes: {original_nc} → {original_names}')

# 找出 "trunk" 類別的原始 class ID
trunk_class_ids = set()
for i, name in enumerate(original_names):
    name_lower = name.lower() if isinstance(name, str) else str(name).lower()
    if 'trunk' in name_lower or 'tree' in name_lower:
        trunk_class_ids.add(i)
        print(f'  ✓ 保留類別 {i}: {name}')
    else:
        print(f'  ✗ 移除類別 {i}: {name}')

if not trunk_class_ids:
    # 如果沒找到包含 trunk/tree 的類別，保留所有類別
    trunk_class_ids = set(range(original_nc))
    print('  ⚠️ 找不到 trunk 類別，保留全部')

# 重映射標籤：只保留 trunk 類別的標註，其餘刪除
total_kept = 0
total_removed = 0
for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(dataset.location, split, 'labels')
    if not os.path.exists(lbl_dir):
        continue
    split_kept = 0
    split_removed = 0
    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith('.txt'):
            continue
        filepath = os.path.join(lbl_dir, lbl_file)
        with open(filepath, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                if class_id in trunk_class_ids:
                    parts[0] = '0'  # 統一為 class 0 = tree_trunk
                    new_lines.append(' '.join(parts))
                    split_kept += 1
                else:
                    split_removed += 1
        with open(filepath, 'w') as f:
            f.write('\n'.join(new_lines) + '\n' if new_lines else '')
    print(f'  {split}: 保留 {split_kept} 個標註, 移除 {split_removed} 個非 trunk 標註')
    total_kept += split_kept
    total_removed += split_removed

print(f'\n總計: 保留 {total_kept}, 移除 {total_removed}')

# 更新 data.yaml
data_config['nc'] = 1
data_config['names'] = ['tree_trunk']

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'\n✓ data.yaml 已更新:')
print(f'  nc: 1')
print(f'  names: [tree_trunk]')
print(f'  path: {data_yaml_path}')

## Step 4：查看幾張標註範例（確認標註品質）

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def visualize_yolo_seg_label(img_path, lbl_path):
    """視覺化 YOLO segmentation 標註"""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    
    overlay = img.copy()
    
    if lbl_path.exists():
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7:  # class + at least 3 points (6 coords)
                    continue
                class_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                
                # 反正規化：YOLO 存的是 0-1 比例,轉回像素
                points = []
                for i in range(0, len(coords), 2):
                    x = int(coords[i] * W)
                    y = int(coords[i+1] * H)
                    points.append([x, y])
                
                pts = np.array(points, np.int32).reshape((-1, 1, 2))
                
                # 半透明綠色填充 = 樹幹區域
                cv2.fillPoly(overlay, [pts], (0, 255, 0))
                # 綠色邊框
                cv2.polylines(img, [pts], True, (0, 255, 0), 2)
    
    # 混合
    result = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)
    return result

# 隨機顯示 6 張標註範例
import random
train_imgs = sorted(Path(dataset.location, 'train', 'images').glob('*'))
samples = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, samples):
    lbl_path = Path(str(img_path).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt')
    vis = visualize_yolo_seg_label(img_path, lbl_path)
    ax.imshow(vis)
    ax.set_title(img_path.name, fontsize=10)
    ax.axis('off')

plt.suptitle('標註範例（綠色 = tree_trunk mask）', fontsize=16)
plt.tight_layout()
plt.show()

print('\n👆 確認綠色區域是否正確覆蓋樹幹')
print('   如果標註有問題，建議在 Roboflow 上修正後重新下載')

## Step 5：開始訓練！

使用 COCO 預訓練的 YOLOv8n-seg 做 **遷移學習**（不是從零訓練）。

T4 GPU 上約需 40-60 分鐘。

In [ ]:
#@title 訓練參數設定 { display-mode: "form" }

EPOCHS = 150 #@param {type:"slider", min:50, max:300, step:10}
BATCH_SIZE = 16 #@param [8, 16, 32] {type:"raw"}
IMAGE_SIZE = 640 #@param [320, 480, 640] {type:"raw"}

print(f'Epochs: {EPOCHS}')
print(f'Batch:  {BATCH_SIZE}')
print(f'ImgSz:  {IMAGE_SIZE}')
print(f'\n預估時間 (T4 GPU): ~{EPOCHS * 0.3:.0f}-{EPOCHS * 0.5:.0f} 分鐘')

In [ ]:
from ultralytics import YOLO
import time

# 從 COCO 預訓練權重開始遷移學習
model = YOLO('yolov8n-seg.pt')
print(f'✓ 載入 YOLOv8n-seg (COCO pretrained)')
print(f'  Parameters: ~3.4M')
print(f'  Base knowledge: 80 COCO classes → fine-tune to 1 class (tree_trunk)')
print()

start_time = time.time()

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    device=0,               # GPU 0
    workers=2,              # Colab worker limit
    patience=30,            # Early stopping
    save=True,
    save_period=10,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    
    # 資料增強 (針對戶外樹木)
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    flipud=0.0,            # 不上下翻轉
    fliplr=0.5,
    mosaic=0.8,
    mixup=0.1,
    erasing=0.1,
    
    # 分割設定
    overlap_mask=True,
    mask_ratio=4,
    
    amp=True,
    close_mosaic=10,
)

elapsed = time.time() - start_time
print(f'\n✓ 訓練完成！耗時 {elapsed/60:.1f} 分鐘')

## Step 6：驗證模型品質

In [ ]:
from ultralytics import YOLO
import os

# 找到最佳模型
best_pt = '/content/runs/segment/train/weights/best.pt'
if not os.path.exists(best_pt):
    # 嘗試其他路徑
    for p in ['/content/runs/segment/train2/weights/best.pt',
              '/content/runs/segment/train3/weights/best.pt']:
        if os.path.exists(p):
            best_pt = p
            break

print(f'Model: {best_pt}')
size_mb = os.path.getsize(best_pt) / 1024 / 1024
print(f'Size: {size_mb:.1f} MB')

# 驗證
model = YOLO(best_pt)
metrics = model.val(data=data_yaml_path)

box_map50 = float(getattr(metrics.box, 'map50', 0))
seg_map50 = float(getattr(metrics.seg, 'map50', 0))
box_p = float(getattr(metrics.box, 'mp', 0))
box_r = float(getattr(metrics.box, 'mr', 0))

print(f'\n{"="*50}')
print(f'  Detection  mAP50: {box_map50:.3f}  P: {box_p:.3f}  R: {box_r:.3f}')
print(f'  Segment    mAP50: {seg_map50:.3f}')
print(f'{"="*50}')

if seg_map50 >= 0.85:
    print('\n✅ 模型品質優秀！可以匯出到手機')
elif seg_map50 >= 0.70:
    print('\n⚠️ 模型品質可以，但建議加更多訓練資料')
else:
    print('\n❌ 品質不足，需要更多資料或更長訓練')

## Step 7：看看模型在真實圖片上的表現

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import random

# 在驗證集上跑推論
val_imgs = sorted(Path(dataset.location, 'valid', 'images').glob('*'))
if not val_imgs:
    val_imgs = sorted(Path(dataset.location, 'test', 'images').glob('*'))

samples = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, samples):
    results = model.predict(str(img_path), verbose=False)
    annotated = results[0].plot()  # 畫出 bbox + mask
    ax.imshow(annotated[:, :, ::-1])  # BGR → RGB
    
    n_det = len(results[0].boxes)
    confs = [f'{c:.2f}' for c in results[0].boxes.conf.cpu().numpy()] if n_det > 0 else []
    ax.set_title(f'{img_path.name}\n{n_det} detections {confs}', fontsize=9)
    ax.axis('off')

plt.suptitle('模型推論結果（藍色 mask = 偵測到的樹幹）', fontsize=16)
plt.tight_layout()
plt.show()

## Step 8：匯出模型

匯出三種格式：
- **TFLite INT8**：手機端 Flutter 使用（~6.3 MB）
- **ONNX**：伺服器端使用
- **PyTorch**：備份

In [ ]:
import shutil

export_dir = '/content/exported_models'
os.makedirs(export_dir, exist_ok=True)

# 1. TFLite INT8
print('匯出 TFLite INT8...')
try:
    tflite_path = model.export(format='tflite', imgsz=IMAGE_SIZE, int8=True)
    if tflite_path:
        shutil.copy2(tflite_path, os.path.join(export_dir, 'tree_trunk_seg.tflite'))
        size = os.path.getsize(tflite_path) / 1024 / 1024
        print(f'  ✓ TFLite: {size:.1f} MB')
except Exception as e:
    print(f'  ✗ TFLite export failed: {e}')
    print('  Trying FP16 fallback...')
    try:
        tflite_path = model.export(format='tflite', imgsz=IMAGE_SIZE, half=True)
        if tflite_path:
            shutil.copy2(tflite_path, os.path.join(export_dir, 'tree_trunk_seg.tflite'))
            size = os.path.getsize(tflite_path) / 1024 / 1024
            print(f'  ✓ TFLite FP16: {size:.1f} MB')
    except Exception as e2:
        print(f'  ✗ TFLite FP16 also failed: {e2}')

# 2. ONNX
print('\n匯出 ONNX...')
try:
    onnx_path = model.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True)
    if onnx_path:
        shutil.copy2(onnx_path, os.path.join(export_dir, 'tree_trunk_seg.onnx'))
        size = os.path.getsize(onnx_path) / 1024 / 1024
        print(f'  ✓ ONNX: {size:.1f} MB')
except Exception as e:
    print(f'  ✗ ONNX export failed: {e}')

# 3. 複製 PyTorch best.pt
shutil.copy2(best_pt, os.path.join(export_dir, 'tree_trunk_seg_best.pt'))
print(f'\n✓ PyTorch: {os.path.getsize(best_pt) / 1024 / 1024:.1f} MB')

# Labels
with open(os.path.join(export_dir, 'tree_trunk_labels.txt'), 'w') as f:
    f.write('tree_trunk\n')
print('✓ Labels file created')

print(f'\n所有模型已匯出到: {export_dir}')
!ls -la {export_dir}

## Step 9：下載模型

打包所有模型成 zip，點擊下載到你的電腦。

下載後：
1. 把 `tree_trunk_seg.tflite` 放到 `frontend/assets/ml/`
2. 把 `tree_trunk_labels.txt` 放到 `frontend/assets/ml/`
3. 把 `tree_trunk_seg.onnx` 放到 `backend/ml_service/models/`（伺服器端用）

In [ ]:
# 打包下載
!cd /content && zip -r exported_models.zip exported_models/

from google.colab import files
files.download('/content/exported_models.zip')
print('\n✓ 下載開始！')
print('\n下載後的部署步驟：')
print('  1. 解壓縮')
print('  2. tree_trunk_seg.tflite → frontend/assets/ml/')
print('  3. tree_trunk_labels.txt → frontend/assets/ml/')
print('  4. tree_trunk_seg.onnx   → backend/ml_service/models/')
print('  5. tree_trunk_seg_best.pt → 保留備份（日後可繼續訓練）')

## Step 10（可選）：用你自己的照片測試

In [ ]:
#@title 上傳照片測試 { display-mode: "form" }
from google.colab import files
import cv2
import matplotlib.pyplot as plt

print('請上傳一張樹幹照片...')
uploaded = files.upload()

for filename, data in uploaded.items():
    # 存檔
    with open(f'/content/{filename}', 'wb') as f:
        f.write(data)
    
    # 推論
    results = model.predict(f'/content/{filename}', verbose=False)
    annotated = results[0].plot()
    
    n_det = len(results[0].boxes)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(f'偵測結果：{n_det} 個樹幹', fontsize=16)
    plt.axis('off')
    plt.show()
    
    if n_det > 0:
        for i, (box, conf) in enumerate(zip(results[0].boxes.xyxy, results[0].boxes.conf)):
            print(f'  樹幹 {i+1}: bbox={box.cpu().numpy().astype(int).tolist()} conf={conf:.3f}')
        if results[0].masks is not None:
            print(f'  Mask pixels: {int(results[0].masks.data.sum())}')
    else:
        print('  未偵測到樹幹')

---
## 訓練完成！

### 你得到了什麼？
- `tree_trunk_seg.tflite` — 手機端模型（~6 MB, 30+ FPS）
- `tree_trunk_seg.onnx` — 伺服器端模型
- `tree_trunk_seg_best.pt` — PyTorch 備份（可以繼續訓練）

### 接下來？
1. 下載模型到電腦
2. 放到對應的目錄
3. 更新 Flutter `tflite_tracking_service.dart` 的推論程式碼
4. 用 `validate_accuracy.py` 驗證 DBH 測量精度

### 想要更準？
- 加入你自己在台灣拍的樹幹照片（用 Roboflow 標註）
- 再訓練一次（這個 notebook 可以重複使用）